bilanciamento dataset


In [ ]:
import torch
from torchvision.datasets import CelebA
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.transforms import v2
import os
import numpy as np

IMAGE_SIZE = 64 # dimensione immagini di output desiderata

transform_pipeline = v2.Compose([
    v2.CenterCrop(140), # Il crop viene applicato prima del resize in modo tale da non deformare il viso per isolare la regione centrale del volto, eliminando background e rumore periferico.
    v2.Resize(IMAGE_SIZE),
    v2.RandomHorizontalFlip(p=0.5), # flip orizzontale per sfruttare la simmetria dei volti umani aiuta il modello a capire posizionamento di occhi ...
    v2.ToImage(), # trasforma l immagine HxWxC in un tensore CxHxW
    v2.ToDtype(torch.float32, scale=True), # prende i valori tra 0 e 255 e li converte in float portandoli tra 0 e 1
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # il batch viene portato ad assumere valori nel range [-1,1] centrato in 0. Questo è coerente col tipo di rumore che sarà introdotto nel forward process, evitando bias nel rumore
])


def make_weights_perfect_8_classes(dataset):
    """
    Divide il dataset in 8 sottoclassi esatte (tutte le combinazioni di
    Male/Not male, Smiling/Not Smiling, Young/Not Young) e assegna pesi affinche
    ogni combinazione abbia la STESSA probabilita di essere estratta.
    """

    # Estraiamo gli attributi (0 o 1)
    # Indici: 20=Male, 31=Smiling, 39=Young
    # li converte a tipo di dato int64
    attrs = dataset.attr[:, [20, 31, 39]].long()

    # 3 attributi come bit di un numero binario a 3 bit
    # labels ha le dimensioni del dataset
    # contiene valori da 0 a 7, per distinguere ogni terna possibile di attributi
    # ogni immagine ottiene un ID in base alla sua configurazione
    labels = attrs[:, 0] * 4 + attrs[:, 1] * 2 + attrs[:, 2]

    # Contiamo quanti elementi esistono per ognuna delle 8 classi
    # restituisce vettore di dimensione 8. Per ogni configurazione possibile esegue conteggio.
    # minlength assicura la dimensione minima a 8. Nel caso assurdo dovessero esserci meno di 8 classi nel dataset
    # si tratta di defensive programming
    class_counts = torch.bincount(labels, minlength=8)

    print("Conteggio classi originali:")
    for i, count in enumerate(class_counts):
        print(f"  Classe {i}: {count.item()} immagini")

    # Piu una classe e rara, piu alto e il peso.
    class_weights = 1.0 / class_counts.float()

    # Gestiamo il caso di classi vuote per evitare inf
    # se una classe non è presente nel batch, non dare peso infinito, ma ignorala per quel batch.
    # defense programming
    class_weights[class_weights == float('inf')] = 0

    # Assegniamo il peso a ogni immagine
    # sample_weights è un vettore di dimensione pari a labels (dimensione del dataset)
    # Operazione di ricerca, viene eseguita un'operazione di 'indexing' (o lookup) vettorizzata:
    # L'indice i di labels determina l'immagine, il contenuto di labels determina la classe (in terna) di quell'immagine
    # da class_weights si preleva il peso dedicato a quella classe in base alla sua frequenza
    # Alla fine si ottiene un vettore di dimensione pari al dataset in cui per ogni immagine c'è il suo peso.
    sample_weights = class_weights[labels]

    # A questo punto ogni combinazione ha probabilita 1/8 di essere pescata nei batch
    return sample_weights

def get_dataloader(root_path, batch_size, num_workers=4):

    # lambda function usata dalla classe CelebA quando esegue internamente __get_item__ per estrarre il dataset.
    # Il metodo prende le immagini con tutti e 40 gli attributi di Celeba
    # La funzione select_attrs definisce una lista di indici per i soli attributi di interesse.
    select_attrs = lambda t: t[[20, 31, 39]]

    # Classe che automatizza l'ingestion dei dati
    # root specifica la posizione del dataset, dove risiedono le immagini
    # split=all vuol dire che la suddivisione del dataset in train val e test viene ignorata.
    # transform specifica le trasformazioni da applicare al volo ad ogni immagine che viene pescata dal path.
    # target_transform permette di utilizzare la lambda function per snellire il dataset ai soli attributi di interesse
    # download false altrimenti scaricherebbe 20 GB di dataset.
    dataset = CelebA(
        root=root_path,
        split='all',
        transform=transform_pipeline,
        target_transform=select_attrs,
        download=False
    )

    # Calcoliamo i pesi personalizzati per ogni immagine
    weights = make_weights_perfect_8_classes(dataset)

    # nativo di pytorch per la gestione dei batch di dati, lavora in simbiosi col DataLoader
    # per scegliere le immagini con cui creare i batch, non utilizza scelte e shuffle casuali ma basa la scelta in base al peso che possiede ciascuna immagine
    # num_samples definisce quante immagini devono essere utilizzate all'interno di una singola epoca, si seleziona la dimensione del dataset
    # replacement = True permette di riutilizzare una stessa immagine nel batch. Altrimenti dopo le prime estrazioni una classe rara non sarebbe più rappresentata per il resto dell'epoca e la situazione sarebbe ancora sbilanciata
    sampler = WeightedRandomSampler(
        weights,
        num_samples=len(weights),
        replacement=True
    )

    # dataloader è un iteratore di pytorch che prepara i dati in modo efficiente per la GPU
    # tra i suoi parametri si specifica il dataset da leggere e il sampler per la costruzione dei batch
    # shuffle flase perchè l'organizzazione dei batch è delegata al sampler
    # batch size specifica il numero di immagini che vengono considerate per volta. maggiore è il batch size, più stabile è il gradiente, maggiore è il consumo di GPU
    # workers = core della CPU che applicano le trasformazioni alle immagini e le preparano in parallelo per la costruzione di altri batch, mentre la GPU processa batch precedenti.
    # drop_last evita la formazione e il processamento di batch con pochi campioni alla fine dell'epoca, più instabili rispetto gli altri
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        sampler=sampler,
        num_workers=num_workers,
        drop_last = True
    )

    return loader

Noise Scheduling

In [ ]:
# Noise scheduling utilizzato nel forward process durante il training per calcolare i campioni rumorosi zt
# utilizzato anche in fase di generazione per ponderare il rumore da togliere ad ogni step di de-noising


# Beta_t: coefficienti di varianza (noise schedule). Rappresentano la frazione di varianza aggiunta allo step t.
# Beta_t e' la quantità di rumore localmente inserito, per passare dal rumore zt-1 al rumore zt. misura quanto si e' degradata l'immagine in questo step.
# 1 - Beta_t: rappresenta la frazione di varianza del segnale che viene preservata nel passaggio da t-1 a t.


# Alpha (nel codice riferito a alpha_bar): è il prodotto cumulato di (1 - Beta).
# Indica la quantità totale di segnale originale (x0) ancora presente dopo t step di rumore.
# La relazione ricorsiva è: alpha_cum_t = (1 - Beta_t) * alpha_cum_t-1

# L indica il numero di step di rumore totali.
# Maggiore è L, più step diamo alla rete per ricostruire l'immagine. Questo permette di utilizzare dei coefficienti Beta molto piccoli.
# Un L ridotto invece costringe la rete a imparare a ricostruire l'immagine z0 = x applicando dei tagli più grossolani al rumore ad ogni step di denoising.
# Avere un L elevato per poter utilizzare dei Beta molto piccoli si sposa con i concetti teorici studiati: la funzione del reverse process q(zt-1|zt) che il modello cerca di imparare si approssima meglio
# ad una gaussiana se nel prodotto di Bayes rendiamo q(zt|zt-1) (forward process governato da Beta e alfa) una campana stretta e alta (da moltiplicare con q(zt-1)).

# Il problema con L elevato è che usando un noise scheduling lineare, già allo step es 200 l'immagine sarà puro rumore.
# Per l'80% del tempo la rete lavorerà solo sul rumore per poi ricreare davvero l'immagine negli ultimi 200 step. --> Manteniamo i Beta piccoli dato L = 1000, ma è inefficiente.
# Con il cosine scheduling invece il rumore si spalma più dolcemente su tutti i 1000 step, permettendo alla rete di lavorare attivamente per ricostruire l'immagine sin dagli step più rumorosi.

import torch
from torchvision.datasets import CelebA
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.transforms import v2
import os
import numpy as np


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class NoiseSchedule:
    def __init__(self, L, s=0.008, device=device):
        self.L=L

        # generazione di una sequenza di 1001 valori equidistanti: 0 1 2 ... 1000
        # L+1 per includere anche 0 e 1000
        # si divide per L per trasformare tutto in un range continuo tra 0 e 1 --> step 500 diventa 0.5
        t=torch.linspace(0.0, L, L+1, device=device)/L


        # a rappresenta la quantità di segnale originale che sopravvive dopo t step di rumore --> alfa
        # al tempo t = 0 immagine perfetta a = 1
        # al tempo t = L rumore puro a = 0
        # viene scelta funzione cos^2 perchè nell'intervallo 0° - 90° ha esattamente questo comportamento
        # il parametro s serve a shiftare un po' l'andamento della curva. Quando t = 0 --> cos^2(t*pi/2) = 1 siamo sulla cresta del coseno --> derivata nulla --> velocità di iniezione del rumore nulla
        # i primi 10 20 step non modificherebbero l'immagine, sprecando risorse computazionali con l'addestramento su quegli step.
        # con il parametro s, l'andamento cosinusoidale non comincia con la cresta, ma un po' sulla discesa. --> questo però non rende più vero  al tempo t = 0 immagine perfetta a = 1
        a=torch.cos((t+s)/(1+s)*torch.pi/2)**2

        # Il primo elemento della sequenza a, deve essere pari a 1: in questo modo ci assicuriamo che all'inizio l'immagine e' preservata interamente, e poi man mano alfa decresce.
        # eseguiamo normalizzazione con primo valore.
        # dato l'utilizzo di s, il primo valore non sarebbe stato 1, ma il valore assunto dal coseno un po' shiftato sulla discesa
        a=a/a[0]

        # se abbiamo a = [a0, a1, a2, a3]
        # a[1:] considera [a1, a2, a3]
        # a[:-1] considera [a0, a1, a2]
        # beta calcolato come (1-a[1:]/a[:-1]), segue la logica specificata sopra: Beta_t = 1 - alpha / alpha_t-1
        # si forma una lista di beta [1-(a1/a0), 1-(a2/a1), 1-(a3/a2)]

        self.beta=(1-a[1:]/a[:-1]).clip(0.0, 0.99) # clip serve a non rendere mai Beta = 1, altrimenti 1 - Beta = 0 e violerei la stabilità numerica.
        self.alpha=torch.cumprod(1.0-self.beta, dim=0)
        self.one_minus_beta=1-self.beta
        self.one_minus_alpha=1-self.alpha
        self.sqrt_alpha=torch.sqrt(self.alpha)
        self.sqrt_beta=torch.sqrt(self.beta)
        self.sqrt_1_alpha=torch.sqrt(self.one_minus_alpha)
        self.sqrt_1_beta=torch.sqrt(self.one_minus_beta)

        # tutte queste variabili hanno dimensione L
        # la rete neurale lavorerà con un indice t che va da 0 a 999, e sarà utilizzato per estrarre coefficienti alfa e beta già precalcolati qui
        # grazie a questo precalcolo, l'operazione di pescaggio dei coefficienti per il rumore t è O(1)

    def __len__(self):
        return self.L

# Number of steps
L=1000
noise_schedule=NoiseSchedule(L)

Time encoding

In [ ]:
import math

# il Noise Scheduling permette di alterare l'immagine in modo organizzato.
# Solo dal noise scheduling la rete non sa precisamente a che step di rumore stiamo lavorando. Vede solo il livello di rumore programmato per un determinato step, senza sapere dove siamo.
# Per far capire alla rete precisamente dove ci troviamo nella sequenza di step, utilizziamo il time encoding.

# In modo deterministico, durante il train, sarà estratto un valore t casuale tra 1 e 1000, ad es t = 5, che sarà utilizzato per ottenere un noise scheduling (con cui costruire zt) e un time encoding corrispondente
# attraverso l'andamento cosinusoidale, allo step 5 ci sarà poco rumore, e ad esso sarà abbinato l'encoding temporale corrispondente.
# Cercando di minimizzare la loss, la rete capirà che con time encoding bassi corrispondenti ai primi step, ci sarà poco rumore aggiunto dal noise scheduling e pertanto dovrà lavorare sul rumore in modo tale da rifinire i dettagli.

# Questa classe viene usata nel metodo forward del modello per ottenere l'encoding dell'istante temporale (quale step dei 1000) corrente.
# Grazie a questa classe la rete riesce a capire in che momento siamo, quale tipo di pulizia e stima deve effettuare del rumore.

# Durante il train viene estratto un numero t tra 1 e 1000.
# Questo numero viene usato come indice numerico nel Noise Scheduling per selezionare i coefficienti alfa corrispondenti ad un certo livello di rumore
# poi di questo valore t, la rete calcola il TimeEncoding per poter identificare quel preciso step di rumore e fondere questa informazione nella rete, così che possa capire dove ci troviamo.


TIME_ENCODING_SIZE=64

class TimeEncoding:
    def __init__(self, L, dim, device=device):
        # Note: the dimension dim should be an even number
        self.L=L # L = 1000
        self.dim=dim #64
        dim2=dim//2 # numero di componenti dell'encoding temporale --> 1 coppia per ciascun elemento.
        encoding=torch.zeros(L, dim) # per ogni step di rumore (1000 in totale), ho 64 colonne che descrivono l'encoding temporale. Encoding di dimensione 1000 righe x 64 colonne
        ang=torch.linspace(0.0, torch.pi/2, L) # 1000 frequenze equamente spaziate tra 0 e 90°
        logmul=torch.linspace(0.0, math.log(40), dim2) # 32 moltiplicatori di frequenze per variare seni e coseni. L'utilizzo di log permette di creare una sequenza logaritmica che permette una migliore copertura delle frequenze rispetto ad una sequenza lineare, in cui tra elementi successivi c'è una grande salto tra i moltiplicatori.
        mul=torch.exp(logmul) # per rendere i moltiplicatori tra 1 e 40. I moltiplicatori più elevati saranno applicati agli ultimi componenti di encoding che serviranno per distinguere tra step consecutivi (a frequenze elevate anche tra t = 450 e t= 451 c'è una grande differenza). I primi invece, più lenti serviranno a distinguere a lungo raggio, tra inizio e fine.
        for i in range(dim2): #per tutte e 32 le coppie di frequenze
            a=ang*mul[i] # moltiplichiamo 1000 valori diversi per un moltiplicatore --> questo rende ogni encoding diverso dagli altri
            encoding[:,2*i]=torch.sin(a) # Calcoliamo il seno di 1000 valori e li mettiamo in una colonna pari
            encoding[:,2*i+1]=torch.cos(a) # calcoliamo coseno di 1000 valori e li mettiamo in una colonna dispari
            # una coppia di embedding lavora ad una certa frequenza, passando alla prossima coppia, cambiamo moltiplicatore e lavoreranno ad una frequenza diversa.
            # prime componenti di embedding lavorano a frequenze basse (moltiplicatore basso) e servono per distinguere gli step a lungo raggio (inizio, metà, fine processo)
            # ultime componenti di embedding lavorano a frequenze elevate (moltiplicatore alto) e servono per distinguere step consecutivi (step 450 - 451).
        self.encoding=encoding.to(device=device)

    def __len__(self):
        return self.L

    def __getitem__(self, t):
        return self.encoding[t]


time_encoding=TimeEncoding(L, TIME_ENCODING_SIZE)

MLP per fusione temporal encoding e informazione condizionata

In [ ]:
import torch
import torch.nn as nn
import math


# Questa classe viene usata dall'architettura per eseguire una fusione semantica
# tra l'informazione condizionata (che durante il training viene fornita alla rete come un tensore di dimensione [Batch, 3]) --> per ogni immagine del batch la terna Male, Smiling, Young
# e il time encoding. --> Informazione da fondere nella rete per farle apprendere in che istante ci troviamo, di dimensione [Batch, 64] --> predispongo un ID per ogni immagine.

# La classe concatena le informazioni allungando le colonne, creando un tensore di dimensione [Batch, 67] --> per ogni immagine

# 64 valori del tempo + 3 valori della condizione provengono da spazi matematici completamente diversi.
# La fusione che espande a CONTEXT_DIM canali permette di ottenere una fusione delle informazioni per dire alla rete
# a che livello di rumore siamo, e verso quale tipo di generazione è necessario dirigere il denoising.


# Costanti globali
TIME_ENCODING_SIZE = 64
COND_FEATURES = 3 # Male, Smiling, Young
CONTEXT_DIM = 256 # Dimensione del vettore sintetizzato dall MLP

class ConditionerMLP(nn.Module):
    def __init__(self, time_dim, cond_dim, out_dim):
        super().__init__()
        # Proiettiamo i segnali in uno spazio comune
        self.mlp = nn.Sequential(
            nn.Linear(time_dim + cond_dim, out_dim), # espande da 67 a CONTEXT_DIM
            nn.SiLU(), # per catturare correlazioni non lineari tra time e cond
            nn.Linear(out_dim, out_dim), # mantiene i canali
            nn.SiLU()
        )

    def forward(self, time_emb, cond):
        # cond deve essere float per l MLP
        cond = cond.to(torch.float32)
        # Uniamo i vettori estendendo il numero di colonne: [Batch, 64] + [Batch, 3] -> [Batch, 67]
        combined = torch.cat([time_emb, cond], dim=1)
        return self.mlp(combined) # Restituisce [Batch, CONTEXT_DIM]

# Struttura UNet solo AdaGN

Giustificazione della presenza del dropout:
Il dropout è stato utilizzato per rafforzare la robustezza della rete.

Il dropout layer costringe ogni layer successivo a non abituarsi mai a determinati attivazioni provenienti dal layer precedente, che potrebbero ripetersi e creare delle micro dipendenze interne.

Nel caso dei diffusion models inoltre, l'input è rumore puro e specificamente per il modello DDIM esso deve essere sempre diverso per garantire una buona diversificazione. L'azione del dropout agisce come una sorta di rumore interno che si somma al rumore esterno e garantisce una diversificazione maggiore. In fase di training, con lo spegnimento di attivazioni casuali, anche con lo stesso input potrebbero essere potenzialmente generate immagini diverse, nonostante usiamo un DDIM. Per l'inferenza invece il vincolo stesso rumore stessa immagine vale.



Inoltre qui stiamo valutando una generazione condizionata.

L'azione del dropout in questo caso potrebbe rendere più difficile la task, perchè la rete deve imparare a comprendere e a generare la condizione nonostante essa possa essere parzialmente disattivata oppure ostacolata da attivazioni che non sono usuali per quella specifica classe.

------------------------------------------------

Giustificazione della presenza della Group Normalization:
La Group Normalization è un tipo di normalizzazione che lavora su ogni immagine singolarmente.
Ogni immagine viene considerata lungo la direzione dei canali e suddivisa in più gruppi lungo questa direzione.
Ad esempio un tensore di dimensione 12x12x64 viene considerato 12x12x8 per volta, per 8 volte.
Ad ogni suddivisione, si calcola media e varianza su 12x12x8 valori e si applica la normalizzazione a questo gruppo.
L utilizzo della Group Normalization garantisce:
controllo della magnitudine dei gradienti per non avere delle esplosioni (che con immagini composte completamente da rumore sono probabili) che manderebbero in crisi i pesi;
valutare l'immagine considerando gruppi di canali per volta permette di concentrarsi su estrazione di determinate feature per volta;
inoltre, includendo pixel diversi possiamo confrontare differenze con altre regioni dell'immagine per quelle specifiche feature estratte.

La Batch Normalization è stata scartata come opzione perchè la normalizzazione sarebbe stata effettuata calcolando media e varianza sull'intero batch che però, per motivi di efficienza, contiene diversi livelli di rumorosità t e quindi la media e la varianza non si sarebbero adeguate al processamento dell'intero batch dato che ogni campione del batch rappresenta un momento di de-noising diverso.

-----------------------------------------------------

Giustificazione blocco ResNet: L'architettura ResNet è stata introdotta principalmente per due motivi:

Introduce delle skip connection interne ad ogni blocco, sia durante la discesa (encoder) che durante la salita (decoder). Queste skip connection interne si comportano diversamente da quelle già presenti nella struttura della UNet: quelle della UNet servono a trasferire l'informazione spaziale in fase di ricostruzione a partire dalla semantica. Le due parti vengono concatenate e combinate; le skip connection dei residual block invece permettono di preservare l'input del blocco intatto come arriva, e processarlo con una serie di convoluzioni. Queste due versioni poi vengono sommate per costruire l'output del singolo blocco.

In questo modo ogni blocco della rete impara la singola trasformazione da applicare all'input, in linea con il concetto di de-noising del modello, ogni suo layer impara come trattare singolarmente ogni ingresso. La somma dei tensori, durante la backpropagation si traduce nella formazione di un gradiente perfettamente preservato dalle operazioni di convoluzioni svolte dal blocco, diminuendo il rischio del Vanishing Gradient.

Il beneficio architetturale del Residual Block consiste nella facilitazione del lavoro di "comprensione" della rete: senza il blocco residuale, un encoder o decoder formati puramente da una sequenza di convoluzioni e funzioni di attivazioni, si sarebbe dovuta adeguare e calibrare qualsiasi tipo di input imparando una combinazione di pesi, che messi insieme come blocco garantivano il processamento di ogni ingresso.

In questo modo invece rendiamo ogni layer più indipendente, ogni layer deve comprendere solo cosa applicare all'input iniziale, e se in un determinato step di denoising, dovesse non eseguire alcuna operazione, capirebbe in fretta di non alterare la soluzione, senza dover configurare una sequenza di layer a gestire sia casi del genere che altri casi di processamento.

In [ ]:
import torch
import torch.nn as nn

# Blocco Residual si occupa solo di trasformare l'ingresso lungo la direzione dei canali, usati sia
# nell'encoder che nel decoder.
class AdaGNResidualBlock(nn.Module):
    """
    Blocco Residuale Avanzato:
    - AdaGN (Adaptive Group Normalization) per iniettare il contesto (gamma, beta).
    """
    def __init__(self, in_channels, out_channels, context_dim, dropout=0.1, num_groups=8):
        super().__init__()

        # 1. Proiezione del Contesto (AdaGN)
        # da intendere come appendice adattabile del conditionerMLP --> proietta i 128 canali del contesto al numero di canali che il layer corrente della Unet sta processando.
        # Riceve context_dim e genera 2 vettori di dimensione in_channels (Scala e Shift). Se stiamo processando immagini con 64 canali, scala e shift avranno 64 canali ciasuno.
        # Lavora sul contesto creato dal conditioner definito prima.
        # 2*in_channels serve perchè per ogni input, devo avere un numero di parametri gamma e beta pari al numero di canali
        # l'mlp fornisce in output una coppia gamma, beta per ogni canale.
        self.context_mlp = nn.Linear(context_dim, 2 * in_channels)


        # Inizializziamo a zero l'ultimo layer dell'MLP.
        # Questo garantisce che all'inizio del training gamma=0 e beta=0 (nessuna modifica).
        # l'immagine viene processata senza che un contesto casuale la alteri.
        # Solo dopo che l'mlp conditioner sia stato addestrato con delle iterazioni di back propagation, inizierà a dare un contributo significativo sull'orientamento statistico dei canali.
        nn.init.zeros_(self.context_mlp.weight)
        nn.init.zeros_(self.context_mlp.bias)

        # 2. Normalizzazione
        # affine=False spegne i parametri gamma e beta che il layer cercherebbe di apprendere di default.
        # in questo caso, utilizzando gamma e beta provenienti dal contesto, utilizziamo dei pesi che cambiano sempre
        # per adeguarsi agli input condizionati in fase di generazione
        # lasciarlo a True avrebbe aggiunto solo una ridondanza.
        self.norm1 = nn.GroupNorm(num_groups, in_channels, affine=False)
        self.act1 = nn.SiLU()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)


        # --- 2. Fase di Rifinitura (Secondo Blocco Convoluzionale) ---
        # Dopo aver iniettato il contesto (tempo/etichette) nella prima metà del blocco,
        # questa seconda metà si occupa di elaborare e consolidare le nuove feature condizionate.
        # Include il Dropout per prevenire l'overfitting (aiuta la rete a non memorizzare a memoria il dataset).

        # Usiamo affine=True perché qui NON applichiamo l'AdaGN.
        # Poiché non stiamo fornendo scale e shift esterni, permettiamo a GroupNorm
        # di apprendere i propri parametri classici (gamma e beta standard) per ri-modulare
        # i dati prima della convoluzione finale.
        self.norm2 = nn.GroupNorm(num_groups, out_channels, affine = True)
        self.act2 = nn.SiLU()
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)


        # --- 5. Shortcut Connection (Skip Connection Interna) ---
        # Affinché sia un VERO blocco residuale, dobbiamo sommare l'input (x) all'output processato (h).
        # Tuttavia, in questo blocco il numero di canali cambia (es. in_channels=64, out_channels=128).
        # Non possiamo sommare tensori con un numero di canali diverso.
        if in_channels != out_channels:
            # Usiamo una convoluzione 1x1 per "proiettare" l'input originale al nuovo numero di canali,
            # senza alterare l'informazione spaziale (Altezza x Larghezza rimane invariata).
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        else:
            # Se i canali sono uguali, passiamo il tensore intatto.
            self.shortcut = nn.Identity()

    def forward(self, x, context_emb):
        # --- A. Normalizzazione e Modulazione (AdaGN) ---
        # Normalizziamo spazialmente i canali
        h = self.norm1(x)

        # Estraiamo gamma (scala) e beta (spostamento) dal contesto
        emb = self.context_mlp(context_emb) # [Batch, 2 * in_channels]
        emb = emb.unsqueeze(-1).unsqueeze(-1) # [Batch, 2 * in_channels, 1, 1] aggiunta di due dimensioni per adeguarlo alla forma dell' immagine.
        gamma, beta = emb.chunk(2, dim=1)     # Viene eseguita divisione a metà (2) lungo la dimensione dei canali (dim=1)

        # ora abbiamo gamma e beta con dimensione [Batch, Canali_input, 1, 1]
        # h invece ha dimensioni [Batch, Canali_input, H, W]

        # Applichiamo la modulazione: h = h * (1 + gamma) + beta
        # nell'operazione gamma e beta grazie al broadcasting vengono clonati virtualmente per ogni pixel dell'input normalizzato h
        # applichiamo media e varianza in base al rumore attuale e all'informazione condizionata
        # la forma 1 + gamma sfrutta l'azzeramento dei pesi del context_mlp definito nell'init.
        # In questo modo abbiamo maggior controllo iniziale, inizialmente h passa così com'e
        # successivamente impara che gamma puo essere usato per amplificare o ridurre la varianza rispetto a 1 e beta
        h = h * (1 + gamma) + beta

        h = self.conv1(self.act1(h))
        h = self.conv2(self.dropout(self.act2(self.norm2(h))))

        return h + self.shortcut(x)

class UNetBlock(nn.Module):
    def __init__(self, size, outer_features, inner_features, context_dim, inner_block=None, dropout=0.1):
        super().__init__()
        self.size = size

        # --- ENCODER ---
        # Il blocco riceve outer_features --> numero di canali dell'input ad es 64
        # e li trasforma in inner_features ad es 128 per il blocco UNet successivo.
        # Il context_dim viene passato a parte per essere processato dal layer lineare che produce gamma e beta nell'AdaGN.
        self.enc_block = AdaGNResidualBlock(outer_features, inner_features, context_dim, dropout)
        # Downsampling separato --> le dimensioni spaziali vengono dimezzate (64x64 --> 32x32)
        self.enc_down = nn.Conv2d(inner_features, inner_features, kernel_size=4, stride=2, padding=1, bias=False)


        # Dopo tutti gli encoder l'input che ora ha 512 canali risale lungo i decoder


        # --- DECODER ---
        self.dec_block = AdaGNResidualBlock(inner_features, outer_features, context_dim, dropout)
        # 2. Upsampling separato --> le dimensioni spaziali vengono raddoppiate
        self.dec_up = nn.ConvTranspose2d(outer_features, outer_features, kernel_size=4, stride=2, padding=1, bias=False)


        # --- COMBINER (AGGIORNATO PER SKIP CONNECTION CLASSICA) ---
        # Ora il combiner deve fondere l'output del decoder (outer_features)
        # con l'output dell'encoder (inner_features).
        # Es: se outer=64 e inner=128, riceverà 192 canali e li riporterà a 64.
        self.combiner = nn.Conv2d(outer_features + inner_features, outer_features, kernel_size=1)
        self.inner = inner_block

    def forward(self, x, context_emb):

        # --- 1. ENCODING ---
        # Passiamo direttamente immagine (x) e vettore piatto (context_emb)
        y_enc = self.enc_block(x, context_emb)
        y = self.enc_down(y_enc) #128x32x32

        # --- 2. BLOCCO INTERNO (Ricorsione) ---
        if self.inner:
            y = self.inner(y, context_emb) # viene rieseguita questa funzione forward per ogni encoder decoder interno alla struttura.

        # --- 3. DECODING ---
        y = self.dec_block(y, context_emb)
        x1 = self.dec_up(y)

        # --- 4. SKIP CONNECTION E OUTPUT ---
        x_out = torch.cat((x1, y_enc), dim=1)
        return self.combiner(x_out)

# ==============================================================================
# CLASSE NETWORK (La UNet Completa)
# ==============================================================================
# Questa classe coordina la trasformazione da immagine -> rumore.
# feat_list indica i canali a ogni livello: [64, 128, 256, 512]
#
# Nuova Filosofia AdaGN:
# Non concateniamo più il contesto (testo/tempo) all'immagine allargando i canali.
# Invece, l'MLP crea un vettore latente (128 dim) che viene "sparato" direttamente
# dentro ogni singolo UNetBlock a tutte le risoluzioni (64x64x64, 32x32x128, 16x16x256, 8x8x512).
# In ogni blocco, questo vettore altera le medie e le varianze dei canali
# esistenti, agendo come un "telecomando" che guida la generazione dei dettagli.
# ==============================================================================

class Network(nn.Module):
    def __init__(self, img_size=64, in_channels=3, feat_list=[64, 128, 256, 512], dropout=0.1):
        super().__init__()
        self.img_size = img_size

        # 1. Istanziamo il ConditionerMLP (Sintetizzatore Globale Tempo + Attributi)
        self.conditioner = ConditionerMLP(
            time_dim=TIME_ENCODING_SIZE,
            cond_dim=COND_FEATURES,
            out_dim=CONTEXT_DIM
        )

        # 2. Pre-processamento: Da RGB (3 canali) ai canali base (es. 64)
        self.pre = nn.Sequential(
            nn.Conv2d(in_channels, feat_list[0], kernel_size=3, padding='same'),
            nn.SiLU()
        )

        # 3. Costruzione ricorsiva della UNet (Istanzia i piani dell'edificio)
        self.unet = self.build_unet(img_size, feat_list, dropout)

        # 4. Post-processamento: Dai canali base (es. 64) torna a RGB (3 canali) per predire il rumore
        self.post = nn.Sequential(
            nn.SiLU(),
            nn.Conv2d(feat_list[0], in_channels, kernel_size=3, padding='same')
        )

    def forward(self, x, t, cond):
        """
        x: corrisponde a zt tensore di rumore di shape [Batch, 3, img_size, img_size]
        t: tensore di indici temporali di shape [Batch,64]
        cond: tensore attributi di shape [Batch, 3] (es. Male, Smiling, Young)
        """
        # A. Recuperiamo gli embedding temporali
        # utilizziamo lo stesso t che nel codice di train ha creato zt immagine rumorosa in ingresso
        time_emb = time_encoding[t]

        # B. Creiamo il vettore di contesto puro (Tempo + Attributi) -> [Batch, CONTEXT_DIM]
        # fusione semantica tra condizione e time encoding
        context_emb = self.conditioner(time_emb, cond)

        # C. Flusso attraverso la rete --> da RGB a 64 canali
        x_base = self.pre(x)

        # Il blocco UNet prende separatamente i canali dell'ingresso e dell'embedding contestuale.
        # I blocchi AdaGN all'interno useranno separatamente context_emb per portarlo alla dimensione dei canali di ogni layer interno della UNet
        # In modo che ogni canale di ingresso sarà pilotato dall'azione dei parametri gamma e beta.
        y_unet = self.unet(x_base, context_emb)

        # Ritorna al dominio RGB
        noise_pred = self.post(y_unet)

        return noise_pred

    def build_unet(self, size, feat_list, dropout):
        # Condizione di ricorsione: finche ci sono almeno 2 elementi in feat_list, scendiamo
        if len(feat_list) > 2:
            # Ricorsione: costruisce il blocco piu interno passando la lista ridotta
            # feat_list[1:] restituisce la lista feat_list senza il primo elemento --> passa solo i layer che rimangono da costruire.
            inner_block = self.build_unet(size // 2, feat_list[1:], dropout)
        else:
            # Siamo arrivati al "fondo" della U-Net (bottleneck)
            inner_block = None

        # Costruiamo il blocco corrente, agganciandogli l'inner_block appena creato
        return UNetBlock(
            size=size,
            outer_features=feat_list[0],
            inner_features=feat_list[1],
            context_dim=CONTEXT_DIM,
            inner_block=inner_block,
            dropout=dropout
        )

# Esecuzione

Training

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision # Aggiunto per salvare la griglia di immagini

# ==========================================
# 0. SETUP DEI PERCORSI E DIRECTORY
# ==========================================
LOCAL_ROOT = '/home/Datasets/CELEBA'
CHECKPOINT_DIR = "ddim_checkpoints/weights"
SAMPLES_DIR = "ddim_checkpoints/generations"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)

epoch_count = 0 # variabile che tiene il conto per tutte le volte che la funzione training_epoch viene eseguita

# funzione che esegue una epoca di addestramento
def training_epoch(dataloader, model, optimizer, noise_schedule, loss_function, device, save_dir):
    # rende visibile la variabile di conteggio, in modo tale che possa aggiornarla e tenerla aggiornata
    global epoch_count

    # model.train() avvisa i layer (come Dropout e BatchNorm/GroupNorm)
    # che siamo in fase di addestramento. Ad esempio, attiva lo spegnimento
    # casuale dei neuroni nel Dropout.
    model.train()

    average_loss = 0.0

    # usare enumerate e step per iterare sul dataloader ci dà la possibilità di sapere quale batch stiamo processando
    # in totale processeremo Numero di Batch = Dimensione del Dataset / Batch Size
    for step, (x, target) in enumerate(dataloader):

        # Il dataloader prepara i batch in parallelo usando i workers
        # x sono i batch --> il dataloader ha eseguito delle trasformazioni definite prima per ritagliare le immagini e renderle 64x64
        # x ha dimensione [Batch, Canali, W, H] = [64, 3, 64, 64]

        # target sono invece le label associate ad ogni immagine
        # target ha dimensione [Batch, Canali] = [64, 3], dove 3 corrisponde ad una terna di attributi

        # 1. to(device): Sposta i dati (tensori) dalla memoria RAM della CPU
        # alla memoria della GPU. E fondamentale per sfruttare l accelerazione hardware.
        x = x.to(device)

        # 2. shape[0]: shape restituisce le dimensioni del tensore (es. [64, 3, 64, 64]).
        # L'indice 0 prende il primo valore, ovvero la dimensione del batch (es. 64 immagini).
        n = x.shape[0]

        ############ CONDIZIONE ##################

        # 3. float(): PyTorch richiede che le operazioni matematiche nelle reti neurali
        # avvengano su numeri decimali a virgola mobile (float32).
        cond = target.to(device).float()

        # 4. torch.rand: Genera un tensore di numeri casuali compresi tra 0 e 1.
        # Passiamo (n,) per avere esattamente un numero casuale per ogni immagine del batch.
        # Otteniamo 64 numeri casuali tra 0 e 1.
        P = 0.2
        u = torch.rand((n,), device=device)

        # Maschera booleana: u < P crea un tensore di valori True/False.
        # Usiamo questo tensore per selezionare solo le righe della condizione 'cond' il cui indice corrisponde
        # a quello del vettore u, in cui la condizione è vera.
        # Sovrascriviamo quelle righe con il null token (-1.0).
        cond[u < P, :] = -1.0

        # Il 20% delle immagini del Batch vengono associate ad una condizione nulla indicata da -1
        # Il modello guarderà le immagini rumorose, alle quali come context viene iniettata l'informazione condizionata cond.
        # Cercando di minimizzare la loss la rete imparerà che con determinate condizioni iniettate, la minimizzazione della loss funzione con determinati step di denoising
        # che sono diretti verso una specifica configurazione facciale.
        # Il 20% delle volte invece, la rete capirà che quando come condizione è iniettata -1, la loss sarà minimizzata qualsiasi volto strutturato essa sia capace di creare.

        ##############################################

        # 5. torch.randint: Genera numeri interi casuali.
        # Estraiamo un indice temporale 't' compreso tra 0 e L-1 per ogni immagine del batch.
        t = torch.randint(0, noise_schedule.L, (n,), device=device)

        # 6. torch.randn_like: "Crea un tensore fatto di rumore normale (gaussiano)
        # che abbia esattamente le stesse dimensioni del tensore 'x'".
        # Cosi non dobbiamo specificare a mano [Batch, Canali, Altezza, Larghezza].
        eps = torch.randn_like(x)

        # 7. view(-1, 1, 1, 1): Questa e l operazione piu potente di PyTorch per il reshaping.
        # Attualmente noise_schedule.sqrt_alpha[t] e un vettore 1D di dimensione [Batch].
        # Non possiamo moltiplicare un vettore 1D per un immagine 4D [Batch, Canali, H, W].
        # 'view' cambia la forma del tensore senza spostare i dati in memoria:
        # - Il '-1' dice a PyTorch: "Calcola tu questa dimensione in base agli elementi che hai" (mantiene il Batch).
        # - Gli '1' aggiungono dimensioni fittizie (Canali=1, Altezza=1, Larghezza=1).
        # Ora il tensore e [Batch, 1, 1, 1]. Questo permette a PyTorch di fare "broadcasting":
        # ovvero spalmare e replicare in automatico lo scalare su tutti i pixel dell immagine.
        # su tutti i pixel sarà applicato lo stesso rumore.
        sqrt_a = noise_schedule.sqrt_alpha[t].view(-1, 1, 1, 1)
        sqrt_1_minus_a = noise_schedule.sqrt_1_alpha[t].view(-1, 1, 1, 1)

        # Inserimento del rumore: combinazione lineare tra immagine originale e rumore puro.
        # applicazione diffusion kernel
        zt = sqrt_a * x + sqrt_1_minus_a * eps

        # 8. Forward pass: inviamo i tensori al modello per ottenere la predizione.
        g = model(zt, t, cond)

        # Calcolo della Loss (Errore Quadratico Medio).
        loss = loss_function(g, eps)

        # 9. optimizer.zero_grad(): Di default PyTorch accumula (somma) i gradienti.
        # Dobbiamo azzerarli all inizio di ogni step, altrimenti calcoleremmo le derivate
        # tenendo conto anche dei dati del batch precedente.
        optimizer.zero_grad()

        # 10. loss.backward(): PyTorch naviga a ritroso
        # il grafo computazionale (catena delle derivate) e calcola quanto ogni singolo
        # parametro della rete ha contribuito all errore.
        loss.backward()

        # 11. clip_grad_norm_: "Taglia" le derivate se il loro valore totale supera 5.0.
        # Previene il problema dei "Gradienti Esplosivi" che causano instabilita numerica (NaN).
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        # 12. optimizer.step(): Applica fisicamente l aggiornamento ai pesi della rete
        # basandosi sui gradienti appena calcolati e sulla formula dell ottimizzatore (es. Adam).
        optimizer.step()

        # 13. Estrazione sicura del valore della loss.
        # detach(): Scollega il tensore dal calcolo dei gradienti (risparmia molta memoria RAM).
        # cpu(): Riporta il tensore dalla memoria GPU alla memoria CPU standard.
        # item(): Converte un tensore PyTorch contenente un solo numero in un normale float Python.
        current_loss = loss.detach().cpu().item()

        if step == 0:
            average_loss = current_loss
        else:
            # average loss mostra il trend vero dell'addestramento
            average_loss = 0.9 * average_loss + 0.1 * current_loss

        # Stampa ogni 100 batch
        if step % 100 == 0:
            print(f"Epoch {epoch_count+1} | Step {step}/{len(dataloader)} | Loss: {average_loss:.4f}")

    # epoca completata tutti i batch sono stati analizzati
    epoch_count += 1
    print(f'Epoch {epoch_count} completed. Average Loss: {average_loss:.4f}')

    # --- FASE DI SALVATAGGIO DEL MODELLO ---
    # 14. model.state_dict(): Crea un dizionario Python che mappa il nome di ogni layer
    # ai suoi parametri attuali (pesi e bias) estratti dalla GPU.
    model_state = model.state_dict()

    # Costruiamo il nome del file inserendo il numero dinamico dell epoca corrente.
    model_filename = f"last_model_epoch_{epoch_count}.pth"

    # Uniamo il percorso della cartella al nome del file (es. "saved_models/last_model_epoch_1.pth").
    save_path = os.path.join(save_dir, model_filename)

    # 15. torch.save(): Serializza (converte in byte) il dizionario e lo scrive fisicamente
    # sul disco fisso. Questo file potra essere ricaricato in futuro per la generazione.
    torch.save(model_state, save_path)

    print(f'Modello salvato con successo in: {save_path}\n')



# ==========================================
# 2. ISTANZIAZIONE E AVVIO DEL TRAINING
# ==========================================

# Impostazioni di base
BATCH_SIZE = 64        # Regola in base alla VRAM della tua GPU (se va in OutOfMemory, scendi a 32 o 16)
NUM_EPOCHS = 500       # Numero totale di epoche
LEARNING_RATE = 1e-4   # Ottimo per i modelli di diffusione

print("1. Preparazione DataLoader...")
# Chiamiamo la tua funzione personalizzata (assicurati di aver eseguito la cella che la definisce)
train_loader = get_dataloader(root_path=LOCAL_ROOT, batch_size=BATCH_SIZE, num_workers=8)

print("2. Inizializzazione Rete, Loss e Ottimizzatore...")
# device e gia stato definito nei tuoi script, lo richiamiamo qui per sicurezza
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Istanziamo la rete con i parametri scelti insieme
model = Network(img_size=64, in_channels=3, feat_list=[64, 128, 256, 512], dropout=0.1).to(device)

# Funzione di costo e Ottimizzatore (AdamW e spesso preferito ad Adam per migliore regolarizzazione)
loss_function = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

print("\n=========================================")
print("3. AVVIO DEL TRAINING LOOP")
print(f"Device utilizzato: {device}")
print(f"Batch per epoca: {len(train_loader)}")
print("=========================================\n")

# Ciclo globale delle epoche
for epoch in range(NUM_EPOCHS):
    training_epoch(
        dataloader=train_loader,
        model=model,
        optimizer=optimizer,
        noise_schedule=noise_schedule, # la classe creata precedentemente
        loss_function=loss_function,
        device=device,
        save_dir=CHECKPOINT_DIR
    )

print("Addestramento completato!")

Definizione funzioni per la generazione

In [ ]:
import torch

# ==========================================
# 1. FUNZIONE MATEMATICA DDIM STEP
# ==========================================
def ddim_step(zt, g, eta, tau_curr, tau_prev, noise_schedule):

    # a_curr corrisponde ad alfa_t
    # a_prev corrisponde ad alfa_t-deltat

    # noise_schedule.alpha contiene gli alpha
    a_curr = noise_schedule.alpha[tau_curr]
    a_prev = noise_schedule.alpha[tau_prev] if tau_prev >= 0 else torch.tensor(1.0, device=zt.device)

    # Calcolo della varianza sigma
    sigma = eta * torch.sqrt((1.0 - a_prev) / (1.0 - a_curr) * (1.0 - a_curr / a_prev))

    # Calcolo dei coefficienti per la combinazione lineare
    c1 = torch.sqrt(a_prev / a_curr)
    c2 = torch.sqrt(1.0 - a_prev - sigma**2) - torch.sqrt(a_prev * (1.0 - a_curr) / a_curr)

    # Aggiunta di rumore (se eta > 0)
    eps = torch.randn_like(zt) if eta > 0.0 else torch.zeros_like(zt)

    # Calcolo del latente allo step precedente
    z_prev = c1 * zt + c2 * g + sigma * eps
    return z_prev

# ==========================================
# 2. FUNZIONE DI GENERAZIONE PRINCIPALE
# ==========================================
@torch.no_grad() # Disattiva il calcolo dei gradienti
def generate_ddim(model, noise_schedule, cond, tau, lam=3.0, eta=0.0, device='cuda'):
    model.eval() # Imposta la rete in modalita' inferenza (spegne i Dropout)

    # viene richiesta la generazione di 32 immagini
    # 8 combinazioni, 4 immagini per ciascuna combinazione
    # n = 32
    n = cond.shape[0]

    # 1. Partiamo da puro rumore gaussiano (3 canali, 64x64)
    z = torch.randn(n, 3, 64, 64, device=device)

    # 2. Creiamo la condizione nulla per il CFG (Classifier-Free Guidance)
    cond0 = torch.full_like(cond, -1.0, device=device)

    # 3. Loop di Denoising al contrario
    # il parametro tau rappresenta gli step di denoising che il modello ddim eseguirà.
    # impostando un delta_t pari a 10, quindi L = 1000 step totali saranno compiuti in 100 passi.
    # len(tau) = 100
    # range(100) crea una lista di indici da 0 a 99
    # reversed fa scorrere la lista da 99 a 0
    for kt in reversed(range(len(tau))):
        tau_curr = tau[kt]
        tau_prev = tau[kt-1] if kt > 0 else -1

        # Mettiamo t sul device corretto
        # espandiamo virtualmente senza allocare VRAM il valore t
        # per indicare lo stesso livello di rumore a tutte le 32 immagini
        t = torch.tensor([tau_curr], device=device).expand(n)

        # Doppio Forward Pass
        g1 = model(z, t, cond)  # Predizione CON gli attributi
        g0 = model(z, t, cond0) # Predizione SENZA attributi

        # Estrapolazione CFG
        g = lam * g1 + (1.0 - lam) * g0

        # Discesa di uno step
        z = ddim_step(z, g, eta, tau_curr, tau_prev, noise_schedule)

    # 4. De-Normalizzazione
    # Le nostre immagini erano tra -1 e 1. Le riportiamo tra 0 e 1 per poterle salvare/visualizzare.
    z = (z + 1.0) / 2.0
    z = torch.clamp(z, 0.0, 1.0) # Taglia eventuali artefatti matematici fuori range

    return z

Generazione

In [ ]:
import os
import torch
import torchvision

# ==========================================
# 0. SETUP DIRECTORY (GENERAZIONE)
# ==========================================
SAMPLES_DIR = "ddim_checkpoints/generations"
os.makedirs(SAMPLES_DIR, exist_ok=True)

# ==========================================
# 1. FUNZIONE DI GENERAZIONE E SALVATAGGIO
# ==========================================
def generate_and_save_grid(model, noise_schedule, device, epoch, save_dir):

  print("Generazione griglia di validazione in corso...")
  # Creiamo le 8 combinazioni possibili per (Male, Smiling, Young)

  combinations = [
      [0, 0, 0], [0, 0, 1], [0, 1, 0], [0, 1, 1],
      [1, 0, 0], [1, 0, 1], [1, 1, 0], [1, 1, 1]
  ]
  # Vogliamo 4 immagini per ogni combinazione (Totale 32 immagini)
  cond_list = []
  for c in combinations:
    for _ in range(4): # 4 varianti (seed diversi) per riga
      cond_list.append(c)

  # Trasformiamo la lista in un tensore [32, 3] e lo mandiamo su GPU
  cond_tensor = torch.tensor(cond_list, dtype=torch.float32, device=device)

  # Prepariamo gli step per il DDIM (100 step saltando ogni 10)
  tau = list(range(0, 1000, 10))

  # Generiamo i volti usando la funzione definita precedentemente
  # Lam=4.0 e un buon compromesso per forzare i tratti senza bruciare i colori
  generated_images = generate_ddim(model, noise_schedule, cond=cond_tensor, tau=tau, lam=4.0, eta=1.0, device=device)

  # Salviamo la griglia. nrow=4 significa che andrà a capo ogni 4 immagini,
  # creando esattamente 8 righe, una per ogni combinazione!
  grid_filename = os.path.join(save_dir, f"generations_epoch_{epoch}.png")
  torchvision.utils.save_image(generated_images, grid_filename, nrow=4, padding=2, normalize=False)

  print(f'Griglia di validazione salvata in: {grid_filename}\n')

# ==========================================
# 2. ESECUZIONE DELLA GENERAZIONE
# ==========================================

# Chiamiamo la funzione (usa l'attuale 'epoch_count' se eseguita subito dopo il train)
generate_and_save_grid(
    model=model,
    noise_schedule=noise_schedule,
    device=device,
    epoch=epoch_count, # Oppure metti manualmente il numero, es. "finale" o 500
    save_dir=SAMPLES_DIR
)